In [1]:
import os
import re
import sys
import json
import torch
import numpy as np
import subprocess
import unicodedata
from pathlib import Path
from datetime import timedelta
from PIL import Image
import soundfile as sf
import scipy.io.wavfile
from scipy.io import wavfile

from sentence_transformers import SentenceTransformer
import open_clip

from pocket_tts import TTSModel

from sqlalchemy.orm import Session
from sqlalchemy import text

import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

c:\Users\lidia\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

SCENE_SIM_THRESHOLD = 0.65
IMAGE_SIM_WEIGHT = 0.3
DESC_SIM_WEIGHT = 0.7
MAX_IMAGE_DURATION = 7

In [4]:
device

'cuda'

In [ ]:
import open_clip

model_path = "C:\\Users\\lidia\\Downloads\\open_clip_pytorch_model.bin"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained=model_path
)

model = model.to(device)
model.eval()


In [5]:
script_path = r'D:\1Graduation project\ECHO\experiments\video_generation\video_generation_pharaohs\Scripts\Ramesses II.txt'
with open(script_path, 'r', encoding='utf-8') as file:
    script = file.read()
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.',
 'Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.',
 "He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers."

In [49]:
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?]) +', text) if s.strip()]

sentences = split_sentences(script)
sentences

['Rameses II was one of ancient Egypt’s greatest pharaohs.',
 'Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.',
 'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.',
 'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.\n\nRameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.',
 'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.',
 "Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.\n\nHe continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength.",
 'Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty betw

In [50]:
sentence_model = SentenceTransformer("all-mpnet-base-v2")
sentence_embeddings = sentence_model.encode(sentences, normalize_embeddings=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 744.65it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

scenes = []
current_scene = [sentences[0]]

for i in range(len(sentences)-1):

    sim = cosine(sentence_embeddings[i], sentence_embeddings[i+1])

    if sim < SCENE_SIM_THRESHOLD:
        scenes.append(" ".join(current_scene))
        current_scene = []

    current_scene.append(sentences[i+1])

if current_scene:
    scenes.append(" ".join(current_scene))

print("Detected scenes:", len(scenes))

Detected scenes: 8


In [52]:
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

Path("tts_Outputs").mkdir(exist_ok=True)

durations = []
scene_audio_files = []

for i, scene in enumerate(scenes):

    audio = tts_model.generate_audio(voice_state, scene)

    output = f"tts_Outputs/scene_{i}.wav"

    scipy.io.wavfile.write(output, tts_model.sample_rate, audio.numpy())

    Fs,data = wavfile.read(output)

    duration = len(data)/float(Fs)
   
    durations.append(duration)
    scene_audio_files.append(output)

    print("Scene",i,"duration",duration)

Scene 0 duration 3.68
Scene 1 duration 7.76
Scene 2 duration 6.72
Scene 3 duration 9.68
Scene 4 duration 7.12
Scene 5 duration 14.0
Scene 6 duration 24.56
Scene 7 duration 5.12


In [53]:
audio_data=[]
samplerate=None
for f in scene_audio_files:
    data,sr = sf.read(f)
    if samplerate is None:
        samplerate=sr
    audio_data.append(data)

combined = np.concatenate(audio_data, axis=0)
sf.write("tts_Outputs/final_audio.wav", combined, samplerate)

In [54]:
from sqlalchemy.orm import Session
from sqlalchemy import text

name = Path(script_path).stem

with Session(engine) as session:
    result = session.execute(
        text("""
            SELECT 
                pi.id,
                pi.image_path,
                pi.image_embedding,
                pi.image_description
            FROM pharaohs_images pi
            JOIN pharaohs p ON pi.pharaoh_id = p.id
            WHERE p.name = :name
        """),
        {"name": name}
    )

    images_data=result.fetchall()

print(f"\nFound {len(images_data)} images for {name}")


Found 31 images for Ramesses II


In [55]:
clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
processed_images=[]
for row in images_data:
    image_id,image_path,image_embedding,image_description=row
    if isinstance(image_embedding,str):
        image_embedding=json.loads(image_embedding)

    image_embedding=np.array(image_embedding)
    image_embedding=image_embedding/np.linalg.norm(image_embedding)

    tokens=clip_tokenizer([image_description]).to(device)

    with torch.no_grad():
        desc_emb=model.encode_text(tokens)
        desc_emb/=desc_emb.norm(dim=-1,keepdim=True)

    desc_emb=desc_emb.cpu().numpy()[0]

    processed_images.append({
        "id": image_id,
        "path": image_path,
        "img_emb": image_embedding,
        "desc_emb": desc_emb
    })

In [56]:
# Global set to track images already used across all scenes
used_image_ids_global = set()
TOP_K = 5

def select_sentence_image(scene_text, scene_dur, processed_images, max_image_duration=MAX_IMAGE_DURATION):
    tokens = clip_tokenizer([scene_text]).to(device)
    with torch.no_grad():
        emb = model.encode_text(tokens)
        emb /= emb.norm(dim=-1, keepdim=True)
    scene_emb = emb.cpu().numpy()[0]

    # Rank images by similarity
    ranked = []
    for img in processed_images:
        image_sim = cosine(scene_emb, img["img_emb"])
        desc_sim = cosine(scene_emb, img["desc_emb"])
        score = IMAGE_SIM_WEIGHT * image_sim + DESC_SIM_WEIGHT * desc_sim
        ranked.append((score, img["id"], img["path"]))
        
    ranked.sort(reverse=True, key=lambda x: x[0])

    # ---------------- SELECT UNIQUE IMAGES GLOBALLY ----------------
    selected_paths = []
  
    for score, img_id, img_path in ranked:
        if img_id not in used_image_ids_global:
            used_image_ids_global.add(img_id)
            return [img_path], [scene_dur]

    return [ranked[0][2]], [scene_dur]



In [57]:

expanded_images = []
expanded_durations = []
sentence_scripts = []

for scene_text, scene_dur in zip(scenes, durations):
    sentences = split_sentences(scene_text)
    n_sentences = len(sentences)
    per_sentence_duration = scene_dur / n_sentences if n_sentences > 0 else scene_dur

    for sent in sentences:
        img_path, img_dur = select_sentence_image(sent, per_sentence_duration, processed_images, MAX_IMAGE_DURATION)
        for img_path, img_dur in zip(img_path, img_dur):
            expanded_images.append(img_path)
            expanded_durations.append(per_sentence_duration)
            sentence_scripts.append(sent)
            filename = Path(img_path).name
            print(f"Image: {filename} | \nScript: {sent}\n")

print("Total images selected:", len(expanded_images))
print("Total durations:", expanded_durations)

Image: Rameses II statue from temple.jpg | 
Script: Rameses II was one of ancient Egypt’s greatest pharaohs.

Image: Map during the reign of Rameses II.jpg | 
Script: Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.

Image: Rameses II in battle.jpg | 
Script: At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.

Image: Rameses II relief.jpg | 
Script: He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.

Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.

Image: Temple of Beit el-Wali.jpg | 
Script: This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.

Image: Abu Simbel.jpg | 
Script: Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.

He continued his fat

In [58]:
'''def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

selected_images=[]
used=set()

for scene in scenes:

    tokens=clip_tokenizer([scene]).to(device)

    with torch.no_grad():

        emb=model.encode_text(tokens)
        emb/=emb.norm(dim=-1,keepdim=True)

    scene_emb=emb.cpu().numpy()[0]

    ranked=[]

    for img in processed_images:

        image_sim=cosine(scene_emb,img["img_emb"])
        desc_sim=cosine(scene_emb,img["desc_emb"])

        score=(IMAGE_SIM_WEIGHT*image_sim)+(DESC_SIM_WEIGHT*desc_sim)

        ranked.append((score,img["id"],img["path"]))

    ranked.sort(reverse=True)

    for score,iid,path in ranked:

        if iid not in used:

            selected_images.append(path)
            used.add(iid)
            break

print("Selected images:",len(selected_images))'''

'def cosine(a,b):\n    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))\n\nselected_images=[]\nused=set()\n\nfor scene in scenes:\n\n    tokens=clip_tokenizer([scene]).to(device)\n\n    with torch.no_grad():\n\n        emb=model.encode_text(tokens)\n        emb/=emb.norm(dim=-1,keepdim=True)\n\n    scene_emb=emb.cpu().numpy()[0]\n\n    ranked=[]\n\n    for img in processed_images:\n\n        image_sim=cosine(scene_emb,img["img_emb"])\n        desc_sim=cosine(scene_emb,img["desc_emb"])\n\n        score=(IMAGE_SIM_WEIGHT*image_sim)+(DESC_SIM_WEIGHT*desc_sim)\n\n        ranked.append((score,img["id"],img["path"]))\n\n    ranked.sort(reverse=True)\n\n    for score,iid,path in ranked:\n\n        if iid not in used:\n\n            selected_images.append(path)\n            used.add(iid)\n            break\n\nprint("Selected images:",len(selected_images))'

In [59]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()

client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as exe:
    image_files = list(exe.map(download_image, enumerate(expanded_images)))

In [60]:
print(expanded_images)
print(expanded_durations)

['data/video_generation/raw/pharaohs_images/Ramesses II/Rameses II statue from temple.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Map during the reign of Rameses II.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Rameses II in battle.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Rameses II relief.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Temple of Beit el-Wali.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Abu Simbel.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Peace treaty of the Hitties and Rameses II.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Rameses II mummy.jpg', 'data/video_generation/raw/pharaohs_images/Ramesses II/Battle of Kadesh.jpg']
[3.68, 7.76, 6.72, 9.68, 7.12, 14.0, 12.28, 12.28, 5.12]


In [61]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))

In [62]:
import re
import unicodedata
from datetime import timedelta

# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 42
MAX_LINES = 2
MIN_DURATION = 1.0


# ---------- HELPERS ----------
def normalize_text(text):

    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",
        "\u200B": "",
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text


def format_timestamp(seconds):

    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):

    words = text.split()

    lines = []
    current_line = ""

    for word in words:

        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word

        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    blocks = []

    for i in range(0, len(lines), MAX_LINES):

        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks


# ---------- MAIN FUNCTION ----------
def generate_srt(text_blocks, durations, output_path):

    assert len(text_blocks) == len(durations), "Text blocks and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for text, duration in zip(text_blocks, durations):

        text = normalize_text(text)

        sentences = split_into_sentences(text)

        chunks = []

        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        if len(chunks) == 0:
            continue

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        if total_chars == 0:
            continue

        for chunk in chunks:

            chunk_chars = len(chunk.replace("\n", ""))

            chunk_duration = max(
                MIN_DURATION,
                (chunk_chars / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_blocks.append(
                f"{subtitle_index}\n"
                f"{format_timestamp(start_time)} --> {format_timestamp(end_time)}\n"
                f"{chunk}\n\n"
            )

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print("SRT file saved:", output_path)

In [63]:
generate_srt(
    scenes,
    durations,
    "tts_Outputs/output_subtitles.srt"
)

SRT file saved: tts_Outputs/output_subtitles.srt


In [64]:
def run_ffmpeg(cmd):
    cmd = [str(c) for c in cmd]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080
):
    total_frames = int(duration * fps)

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)

    scaled_w = int(img_w * scale_factor)
    scaled_h = int(img_h * scale_factor)

    max_x = scaled_w - target_w
    max_y = scaled_h - target_h

    threshold = 40  # pixels — below this we zoom instead of pan

    # Smooth easing expression
    #ease = f"(0.5-0.5*cos(PI*t/{duration}))"
    ease = f"(0.5-0.5*cos(PI*n/{total_frames}))"
    
    # Decide motion
    if (img_w / img_h) > (target_w / target_h):
        # Wider image → horizontal movement possible
        if max_x > threshold:
            x_expr = f"floor({max_x}*{ease})"
            y_expr = f"{max_y}/2"
            zoom_filter = ""
        else:
            # Not enough horizontal space → subtle zoom
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"
    else:
        # Taller image → vertical movement
        if max_y > threshold:
            x_expr = f"{max_x}/2"
            y_expr = f"floor({max_y}*{ease})"
            zoom_filter = ""
        else:
            # Not enough vertical space → subtle zoom
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"

    vf = (
        f"scale={scaled_w}:{scaled_h}"
        f"{zoom_filter},"
        f"crop={target_w}:{target_h}:"
        f"x='{x_expr}':"
        f"y='{y_expr}'"
    )

    cmd = [
        "ffmpeg",
        "-y",
        "-loop", "1",
        "-framerate", str(fps),
        "-t", str(duration),
        "-i", image_path,
        "-vf", vf,
        "-frames:v", str(total_frames),
        "-c:v", "libx264",
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-vsync", "cfr",
        #"-video_track_timescale", "30",
        output_path
    ]

    run_ffmpeg(cmd)
    
    
'''    "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",'''

'    "-c:v", "h264_nvenc",\n        "-preset", "p1",\n        "-pix_fmt", "yuv420p",'

In [65]:
def generate_all_clips(image_files, expanded_durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)
    outputs = []
    for i, (img, dur) in enumerate(zip(image_files, expanded_durations)):
        out = f"{temp_dir}/clip_{i}.mp4"
        create_kenburns_clip(img, dur, out)
        outputs.append(out)
    return outputs


def concatenate_clips(clips, output_path):
    list_file = "tts_Outputs/concat_list.txt"

    with open(list_file, "w") as f:
        for clip in clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    cmd = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c:v", "libx264",
        "-preset", "fast",
        #"-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

In [66]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c:v", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "libx264",
        "-preset", "fast",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('tts_Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('tts_Outputs', f))

In [67]:
print(image_files)

[WindowsPath('temp_frames/0000.jpg'), WindowsPath('temp_frames/0001.jpg'), WindowsPath('temp_frames/0002.jpg'), WindowsPath('temp_frames/0003.jpg'), WindowsPath('temp_frames/0004.jpg'), WindowsPath('temp_frames/0005.jpg'), WindowsPath('temp_frames/0006.jpg'), WindowsPath('temp_frames/0007.jpg'), WindowsPath('temp_frames/0008.jpg')]


In [68]:
print(image_files)
import subprocess

subprocess.run(["ffmpeg", "-version"], check=True)

[WindowsPath('temp_frames/0000.jpg'), WindowsPath('temp_frames/0001.jpg'), WindowsPath('temp_frames/0002.jpg'), WindowsPath('temp_frames/0003.jpg'), WindowsPath('temp_frames/0004.jpg'), WindowsPath('temp_frames/0005.jpg'), WindowsPath('temp_frames/0006.jpg'), WindowsPath('temp_frames/0007.jpg'), WindowsPath('temp_frames/0008.jpg')]


CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

In [69]:
clips = generate_all_clips(image_files, expanded_durations)
concatenated_video = "tts_Outputs/combined.mp4"
concatenate_clips(clips, concatenated_video)

video_with_audio = "tts_Outputs/with_audio.mp4"
add_audio(concatenated_video, "tts_Outputs/final_audio.wav", video_with_audio)

generate_srt(scenes, durations, "tts_Outputs/output_subtitles.srt")

final_output = "tts_Outputs/final_video.mp4"
add_subtitles(video_with_audio, "tts_Outputs/output_subtitles.srt", final_output)

cleanup_files()
print("Video generation finished!")
print("Final video:", final_output)

Running: ffmpeg -y -loop 1 -framerate 30 -t 3.68 -i temp_frames\0000.jpg -vf scale=1920:1433,crop=1920:1080:x='0/2':y='floor(353*(0.5-0.5*cos(PI*n/110)))' -frames:v 110 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_0.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 7.76 -i temp_frames\0001.jpg -vf scale=1920:3143,crop=1920:1080:x='0/2':y='floor(2063*(0.5-0.5*cos(PI*n/232)))' -frames:v 232 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_1.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 6.72 -i temp_frames\0002.jpg -vf scale=1920:1268,crop=1920:1080:x='0/2':y='floor(188*(0.5-0.5*cos(PI*n/201)))' -frames:v 201 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_2.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 9.68 -i temp_frames\0003.jpg -vf scale=1920:1920,crop=1920:1080:x='0/2':y='floor(840*(0.5-0.5*cos(PI*n/290)))' -frames:v 290 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_3.mp4
Running: ffmpeg -y 